# Adjacency Sparse Matrix to Cosmograph CSV Files

**Input:** `inter_to_ict_chb01_03_2980_3010_adjacency_sparse.npz`
**Output files:**
1. CZ visualization during 30 sec : `cosmograph_nodes_CZ.csv` + `cosmograph_edges_CZ.csv`
2. Static Graph between Pre- and Ictal : `cosmo_freq_nodes_preictal/ictal.csv` + `cosmo_freq_edges_preictal/ictal.csv`
3. all nodes visualization during 30 sec : `cosmograph_nodes_all.csv` + `cosmograph_edges_all.csv`

In [ ]:
import numpy as np
import math, csv, os
from scipy import sparse
from scipy.sparse import csr_matrix

# For Google Colab: upload the npz file first
# from google.colab import files
# files.upload()  # upload inter_to_ict_chb01_03_2980_3010_adjacency_sparse.npz

In [ ]:
# Load Adjacency Sparse Matrix
N_ELEC  = 23      # number of EEG electrodes
N_TIME  = 7680    # total timepoints (30s × 256Hz)
WINDOW  = 256     # timepoints per 1-second window
N_WIN   = N_TIME // WINDOW   # 30 windows
SEIZURE = 15      # seizure onset window (t=15s)

LABELS = ['FP1','FP2','F7','F3','FZ','F4','F8','T7','C3','CZ','C4','T8',
          'P7','P3','PZ','P4','P8','O1','OZ','O2','A1','A2','T9']

ELEC_COLORS = [
    '#e74c3c','#3498db','#2ecc71','#9b59b6','#f39c12',
    '#1abc9c','#e67e22','#16a085','#8e44ad','#2980b9',
    '#27ae60','#d35400','#c0392b','#7f8c8d','#2c3e50',
    '#f1c40f','#e91e63','#00bcd4','#ff5722','#795548',
    '#607d8b','#9c27b0','#4caf50'
]

# 10-20 head positions (normalized 0~1) for each electrode
ELEC_POS = [
    [.5,.05],[.55,.05],[.15,.18],[.35,.15],[.5,.12],[.65,.15],[.85,.18],
    [.1,.45],[.3,.38],[.5,.35],[.7,.38],[.9,.45],
    [.1,.7],[.3,.62],[.5,.6],[.7,.62],[.9,.7],
    [.3,.85],[.5,.82],[.7,.85],
    [.05,.55],[.95,.55],[.05,.75]
]

# Load sparse matrix
NPZ_PATH = 'inter_to_ict_chb01_03_2980_3010_adjacency_sparse.npz'
mat = sparse.load_npz(NPZ_PATH)
cx = mat.tocoo()  # COO format: row, col, data arrays

print(f"Matrix shape: {mat.shape}")
print(f"Non-zero elements: {mat.nnz:,}")

#  Decode node indices
# Node k → electrode = k // N_TIME, timepoint = k % N_TIME
rows_elec = cx.row // N_TIME   # electrode index of source node
cols_elec = cx.col // N_TIME   # electrode index of target node
rows_time = cx.row % N_TIME    # timepoint of source node
cols_time = cx.col % N_TIME    # timepoint of target node

# Separate intra vs cross electrode edges
same = rows_elec == cols_elec  # True = intra-electrode (actual VG edges)
cross = ~same                  # True = cross-electrode

print(f"Intra-electrode edges (VG): {same.sum():,}")
print(f"Cross-electrode edges:      {cross.sum():,}")

In [ ]:
# compute VG degree from every nodes
# Build intra-electrode adjacency matrix
intra_mat = csr_matrix(
    (np.ones(same.sum()), (cx.row[same], cx.col[same])),
    shape=mat.shape
)

# Compute degree: sum of row + column (undirected graph)
intra_deg = (
    np.array(intra_mat.sum(axis=1)).flatten() +
    np.array(intra_mat.sum(axis=0)).flatten()
)

# Reshape to (electrode, timepoint)
deg_full = intra_deg.reshape(N_ELEC, N_TIME)  # shape: (23, 7680)

print(f"Degree matrix shape: {deg_full.shape}")
print(f"  → 23 electrodes × 7,680 timepoints")
print()
print(f"Pre-ictal mean degree: {deg_full[:, :SEIZURE*WINDOW].mean():.4f}")
print(f"Ictal mean degree:     {deg_full[:, SEIZURE*WINDOW:].mean():.4f}")
print()
print("Per-electrode ictal mean degree (top 5):")
ict_means = deg_full[:, SEIZURE*WINDOW:].mean(axis=1)
for ei in np.argsort(ict_means)[::-1][:5]:
    print(f"  {LABELS[ei]}: {ict_means[ei]:.3f}")

In [ ]:
# visaulzation for CZ electrode only, 30 windows, circular layout, Cosmograph timeline
# Parameters
EI_CZ = 9   # CZ electrode index
R     = 500  # radius of circular layout

# Node circular positions (same for all windows)
node_pos = {}
for k in range(WINDOW):
    angle = (k / WINDOW) * 2 * math.pi - math.pi / 2
    node_pos[k] = (
        round(R * math.cos(angle), 2),
        round(R * math.sin(angle), 2)
    )

# Build nodes
nodes_cz = []
for w in range(N_WIN):
    t_start = w * WINDOW
    phase = 'pre-ictal' if w < SEIZURE else 'ictal'
    for k in range(WINDOW):
        tp = t_start + k
        d  = float(deg_full[EI_CZ, tp])
        x, y = node_pos[k]
        nodes_cz.append({
            'id':        f"w{w}_t{k}",
            'x':         x,
            'y':         y,
            'degree':    round(d, 4),
            'time':      w,          # ← Cosmograph timeline
            'phase':     phase,
            'timepoint': k
        })

# Build edges
edges_cz = []
for w in range(N_WIN):
    t_start = w * WINDOW
    t_end   = (w + 1) * WINDOW

    # Get all intra-electrode edges for CZ in this window
    mask = (
        same &
        (rows_elec == EI_CZ) &
        (rows_time >= t_start) &
        (rows_time < t_end)
    )
    r_local = (cx.row[mask] % N_TIME) - t_start  # 0~255
    c_local = (cx.col[mask] % N_TIME) - t_start  # 0~255

    seen = set()
    for ri, ci in zip(r_local.tolist(), c_local.tolist()):
        if ri >= ci or ri < 0 or ci >= WINDOW:
            continue
        if (ri, ci) in seen:
            continue
        seen.add((ri, ci))
        dist = ci - ri
        edges_cz.append({
            'source': f"w{w}_t{ri}",
            'target': f"w{w}_t{ci}",
            'weight': round(1 / dist, 4),  # closer = heavier
            'time':   w                     # ← Cosmograph timeline
        })

print(f"CZ nodes: {len(nodes_cz):,}  ({N_WIN} windows × {WINDOW} timepoints)")
print(f"CZ edges: {len(edges_cz):,}")
print(f"  Pre-ictal edges: {sum(1 for e in edges_cz if e['time'] < SEIZURE)}")
print(f"  Ictal edges:     {sum(1 for e in edges_cz if e['time'] >= SEIZURE)}")

In [ ]:
# Save File 1
with open('cosmograph_nodes_CZ.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['id','x','y','degree','time','phase','timepoint'])
    w.writeheader(); w.writerows(nodes_cz)

with open('cosmograph_edges_CZ.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['source','target','weight','time'])
    w.writeheader(); w.writerows(edges_cz)

print(f"Saved: cosmograph_nodes_CZ.csv  ({os.path.getsize('cosmograph_nodes_CZ.csv')//1024} KB)")
print(f"Saved: cosmograph_edges_CZ.csv  ({os.path.getsize('cosmograph_edges_CZ.csv')//1024} KB)")

# ── Colab download ──
# try:
    #from google.colab import files
    #files.download('cosmograph_nodes_CZ.csv')
    #files.download('cosmograph_edges_CZ.csv')
#except:
    #print("(Not in Colab — files saved locally)")

In [ ]:
# static visaulization from all 23 electrodes, 15s frequency-weighted average
def build_freq_files(phase):
    """
    Build frequency-weighted VG for all 23 electrodes.
    phase: 'pre' (windows 0-14) or 'ict' (windows 15-29)
    """
    is_pre  = (phase == 'pre')
    w_start = 0       if is_pre else SEIZURE
    w_end   = SEIZURE if is_pre else N_WIN
    n_wins  = w_end - w_start  # 15

    nodes = []
    edges = []
    CANVAS = 5000
    R_ELEC = 80  # radius of each electrode's circular VG

    for ei in range(N_ELEC):
        # Electrode center position (head layout)
        ex = ELEC_POS[ei][0] * CANVAS
        ey = ELEC_POS[ei][1] * CANVAS

        # Step A: Mean degree across 15 windows
        if is_pre:
            avg_deg = deg_full[ei, :SEIZURE*WINDOW].reshape(n_wins, WINDOW).mean(axis=0)
        else:
            avg_deg = deg_full[ei, SEIZURE*WINDOW:].reshape(n_wins, WINDOW).mean(axis=0)

        max_deg = avg_deg.max() if avg_deg.max() > 0 else 1

        # Build nodes (one per timepoint, 256 per electrode)
        for k in range(WINDOW):
            angle = (k / WINDOW) * 2 * math.pi - math.pi / 2
            x = round(ex + R_ELEC * math.cos(angle), 2)
            y = round(ey + R_ELEC * math.sin(angle), 2)
            nodes.append({
                'id':        f"e{ei}_t{k}",
                'x':         x,
                'y':         y,
                'degree':    round(float(avg_deg[k]), 4),   # mean over 15 windows
                'electrode': LABELS[ei],
                'color':     ELEC_COLORS[ei],
                'timepoint': k
            })

        # Step B: Count edge frequency across 15 windows
        edge_count = {}  # (ri, ci) → how many windows had this edge

        for w in range(w_start, w_end):
            t_start = w * WINDOW
            t_end   = (w + 1) * WINDOW

            mask = (
                same &
                (rows_elec == ei) &
                (rows_time >= t_start) &
                (rows_time < t_end)
            )
            r_local = (cx.row[mask] % N_TIME) - t_start
            c_local = (cx.col[mask] % N_TIME) - t_start

            seen = set()
            for ri, ci in zip(r_local.tolist(), c_local.tolist()):
                if ri >= ci or ri < 0 or ci >= WINDOW: continue
                if (ri, ci) in seen: continue
                seen.add((ri, ci))
                edge_count[(ri, ci)] = edge_count.get((ri, ci), 0) + 1

        # Build edges with frequency weight
        for (a, b), cnt in edge_count.items():
            freq = round(cnt / n_wins, 4)  # normalize: count / 15
            edges.append({
                'source':    f"e{ei}_t{a}",
                'target':    f"e{ei}_t{b}",
                'weight':    freq,           # ← frequency (0~1)
                'distance':  b - a,
                'electrode': LABELS[ei]
            })

    return nodes, edges

# Build pre-ictal
nodes_pre, edges_pre = build_freq_files('pre')
print(f"Pre-ictal → nodes: {len(nodes_pre):,}  edges: {len(edges_pre):,}")

# Build ictal
nodes_ict, edges_ict = build_freq_files('ict')
print(f"Ictal     → nodes: {len(nodes_ict):,}  edges: {len(edges_ict):,}")

print()
print(f"Edge increase: {len(edges_ict)/len(edges_pre):.1f}x  ({len(edges_pre)} → {len(edges_ict)})")

In [ ]:
# Save File 2
node_fields = ['id','x','y','degree','electrode','color','timepoint']
edge_fields = ['source','target','weight','distance','electrode']

for name, data in [
    ('cosmo_freq_nodes_preictal.csv', nodes_pre),
    ('cosmo_freq_nodes_ictal.csv',    nodes_ict),
]:
    with open(name, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=node_fields)
        w.writeheader(); w.writerows(data)
    print(f"Saved: {name}  ({os.path.getsize(name)//1024} KB)")

for name, data in [
    ('cosmo_freq_edges_preictal.csv', edges_pre),
    ('cosmo_freq_edges_ictal.csv',    edges_ict),
]:
    with open(name, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=edge_fields)
        w.writeheader(); w.writerows(data)
    print(f"Saved: {name}  ({os.path.getsize(name)//1024} KB)")

# Colab download
#try:
    #from google.colab import files
    #for fname in ['cosmo_freq_nodes_preictal.csv','cosmo_freq_edges_preictal.csv',
       #           'cosmo_freq_nodes_ictal.csv','cosmo_freq_edges_ictal.csv']:
       # files.download(fname)
#except:
    #print("(Not in Colab — files saved locally)")

In [ ]:
# visualization for all 23 electrodes, 30 windows, head layout, Cosmograph timeline
CANVAS_ALL = 5000
R_ALL      = 80   # radius of each electrode's circular VG

nodes_all = []
edges_all = []

for w in range(N_WIN):
    t_start = w * WINDOW
    t_end   = (w + 1) * WINDOW
    phase   = 'pre-ictal' if w < SEIZURE else 'ictal'

    for ei in range(N_ELEC):
        # Electrode center (head layout)
        ex = ELEC_POS[ei][0] * CANVAS_ALL
        ey = ELEC_POS[ei][1] * CANVAS_ALL

        # Nodes: 256 timepoints in a circle around electrode center
        for k in range(WINDOW):
            tp    = t_start + k
            d     = float(deg_full[ei, tp])
            angle = (k / WINDOW) * 2 * math.pi - math.pi / 2
            x     = round(ex + R_ALL * math.cos(angle), 2)
            y     = round(ey + R_ALL * math.sin(angle), 2)

            nodes_all.append({
                'id':        f"e{ei}_w{w}_t{k}",
                'x':         x,
                'y':         y,
                'degree':    round(d, 4),
                'time':      w,              # Cosmograph timeline
                'phase':     phase,
                'electrode': LABELS[ei],
                'color':     ELEC_COLORS[ei],
                'timepoint': k
            })

        # Edges: VG edges for this electrode in this window
        mask = (
            same &
            (rows_elec == ei) &
            (rows_time >= t_start) &
            (rows_time < t_end)
        )
        r_local = (cx.row[mask] % N_TIME) - t_start
        c_local = (cx.col[mask] % N_TIME) - t_start

        seen = set()
        for ri, ci in zip(r_local.tolist(), c_local.tolist()):
            if ri >= ci or ri < 0 or ci >= WINDOW: continue
            if (ri, ci) in seen: continue
            seen.add((ri, ci))
            dist = ci - ri
            edges_all.append({
                'source':    f"e{ei}_w{w}_t{ri}",
                'target':    f"e{ei}_w{w}_t{ci}",
                'weight':    round(1 / dist, 4),
                'time':      w,              # Cosmograph timeline
                'electrode': LABELS[ei]
            })

    if w % 5 == 0:
        print(f"  Window {w:2d} done — nodes so far: {len(nodes_all):,}")

print(f"\nTotal nodes: {len(nodes_all):,}  (23 × 30 × 256)")
print(f"Total edges: {len(edges_all):,}")

In [ ]:
# Save File
node_fields_all = ['id','x','y','degree','time','phase','electrode','color','timepoint']
edge_fields_all = ['source','target','weight','time','electrode']

with open('cosmograph_nodes_all.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=node_fields_all)
    w.writeheader(); w.writerows(nodes_all)
print(f"Saved: cosmograph_nodes_all.csv  ({os.path.getsize('cosmograph_nodes_all.csv')//1024//1024} MB)")

with open('cosmograph_edges_all.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=edge_fields_all)
    w.writeheader(); w.writerows(edges_all)
print(f"Saved: cosmograph_edges_all.csv  ({os.path.getsize('cosmograph_edges_all.csv')//1024} KB)")

# Colab download
#try:
    #from google.colab import files
    #files.download('cosmograph_nodes_all.csv')
    #files.download('cosmograph_edges_all.csv')
#except:
    #print("(Not in Colab — files saved locally)")